# 🔬 Palynology YOLOv26 Segmentation Training
**Sınıflar:** AOM, Background, Palynomorph, Phytoclast

### Başlamadan önce:
1. **Runtime → Change runtime type → T4 GPU** seç
2. Sol panelde 📁 simgesine tıkla, `dataset_split.zip` dosyasını yükle

In [1]:
# ── Google Drive bağla ────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
# Drive'da klasör oluştur
os.makedirs('/content/drive/MyDrive/palytoae', exist_ok=True)
print('✅ Drive bağlandı: /content/drive/MyDrive/palytoae')

Mounted at /content/drive
✅ Drive bağlandı: /content/drive/MyDrive/palytoae


In [ ]:
# ── ADIM 1: GPU kontrolü ──────────────────────────────────────
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ GPU YOK!')
print('CUDA:', torch.version.cuda)
!nvidia-smi

GPU: Tesla T4
CUDA: 12.8
Fri Jun  5 20:12:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+----------------------

In [ ]:
# ── ADIM 2: Ultralytics kur ───────────────────────────────────
!pip install -U ultralytics -q
from ultralytics import YOLO
print('✅ Ultralytics hazır')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Ultralytics hazır


In [ ]:
# ── ADIM 3: Dataset zip'ini çıkart ───────────────────────────
import zipfile, os

ZIP_PATH = '/content/dataset_split.zip'  # Yüklediğin zip
EXTRACT  = '/content/dataset_split'

if not os.path.exists(ZIP_PATH):
    print('❌ dataset_split.zip bulunamadı!')
    print('   Sol panelden dosyayı yükle, sonra bu hücreyi tekrar çalıştır.')
else:
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT)
    print('✅ Zip çıkartıldı:', EXTRACT)

    # Klasör yapısını göster
    for root, dirs, files in os.walk(EXTRACT):
        level = root.replace(EXTRACT, '').count(os.sep)
        indent = '  ' * level
        print(f'{indent}{os.path.basename(root)}/')
        if level < 2:
            for f in files[:3]:
                print(f'{indent}  {f}')

✅ Zip çıkartıldı: /content/dataset_split
dataset_split/
  dataset_split/
    split_report.txt
    dataset_split.rar
    data.yaml
    train/
      images/
      labels/
    valid/
      images/
      labels/
    test/
      images/
      labels/


In [ ]:
# ── ADIM 4: data.yaml yaz ─────────────────────────────────────
import yaml
import os # os modülünü ekledim

data_yaml = {
    'path' : '/content/dataset_split/dataset_split', # Klasör yolunu güncelledim
    'train': 'train/images',
    'val'  : 'valid/images',
    'test' : 'test/images',
    'nc'   : 4,
    'names': ['AOM', 'Background', 'Palynomorph', 'Phytoclast']
}

yaml_path = '/content/dataset_split/data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

# Kontrol
for split in ['train', 'valid', 'test']:
    p = f'/content/dataset_split/dataset_split/{split}/images' # Kontrol yolunu güncelledim
    n = len(os.listdir(p)) if os.path.exists(p) else 0
    print(f'  {split:6s}: {n} görüntü  {"✅" if n > 0 else "❌"}')

print('\n✅ data.yaml hazır')

  train : 400 görüntü  ✅
  valid : 82 görüntü  ✅
  test  : 82 görüntü  ✅

✅ data.yaml hazır


In [ ]:
# ── ADIM 5: TensorBoard başlat ────────────────────────────────
%load_ext tensorboard
%tensorboard --logdir /content/runs

In [ ]:
from ultralytics import YOLO
import time

model = YOLO('yolo26l-seg.pt')  # Colab T4 için m yeterli, l de olur

start = time.time()

results = model.train(
    data    = '/content/dataset_split/data.yaml',
    project = '/content/drive/MyDrive/palytoae' , '/content/runs',
    name    = 'palytoae',

    # Temel
    epochs        = 300,
    patience      = 60,
    batch         = 16,
    imgsz         = 640,
    device        = 0,

    # Optimizer
    optimizer     = 'AdamW',
    lr0           = 0.001,
    lrf           = 0.01,
    cos_lr        = True,
    warmup_epochs = 5,
    close_mosaic  = 15,

    # Loss — AOM/Phytoclast ayrımı için cls artırıldı
    cls           = 2.0,
    box           = 7.5,
    dfl           = 1.5,

    # Augmentation — copy_paste AOM için kritik
    hsv_h         = 0.015,
    hsv_s         = 0.7,
    hsv_v         = 0.4,
    degrees       = 10.0,
    flipud        = 0.5,
    fliplr        = 0.5,
    mosaic        = 1.0,
    copy_paste    = 0.4,
    mixup         = 0.15,

    # Kayıt
    save_period   = 10,
    plots         = True,
    seed          = 42,
    workers       = 2,
)

elapsed = time.time() - start
print(f'\n✅ Training tamamlandı — {int(elapsed//3600)}s {int((elapsed%3600)//60)}dk')
d = results.results_dict
print(f'  mAP50(M)    : {d.get("metrics/mAP50(M)", "N/A")}')
print(f'  mAP50-95(M) : {d.get("metrics/mAP50-95(M)", "N/A")}')
print(f'  Precision   : {d.get("metrics/precision(M)", "N/A")}')
print(f'  Recall      : {d.get("metrics/recall(M)", "N/A")}')

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=2.0, cls_pw=0.0, compile=False, conf=None, copy_paste=0.4, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/dataset_split/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolo26l-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=palytoae, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, pa

In [ ]:
# ── ADIM 7: Modeli indir ──────────────────────────────────────
from google.colab import files
import glob

# En iyi modeli bul
best_models = glob.glob('/content/runs/palytoae*/weights/best.pt')
if best_models:
    best = sorted(best_models)[-1]
    print(f'İndiriliyor: {best}')
    files.download(best)
else:
    print('best.pt bulunamadı, tüm weights klasörünü kontrol et:')
    !find /content/runs -name '*.pt'

In [ ]:
# ── ADIM 8: Per-class değerlendirme ──────────────────────────
import glob

best_models = glob.glob('/content/runs/palytoae*/weights/best.pt')
best = sorted(best_models)[-1] if best_models else None

if not best:
    print('❌ best.pt bulunamadı')
else:
    model = YOLO(best)
    metrics = model.val(
        data  = '/content/dataset_split/data.yaml',
        split = 'test',
        conf  = 0.25,
        iou   = 0.5,
        plots = True
    )

    class_names = ['AOM', 'Background', 'Palynomorph', 'Phytoclast']
    print('\n' + '='*55)
    print('PER-CLASS mAP50 (Segmentation)')
    print('='*55)

    try:
        maps50 = getattr(metrics.seg, 'maps50', metrics.seg.maps)
        for i, name in enumerate(class_names):
            if i < len(maps50):
                val = maps50[i]
                bar = '█' * int(val * 20)
                flag = '✅' if val >= 0.75 else '⚠️ '
                print(f'  {flag} {name:15s}: {val:.3f}  {bar}')
    except Exception as e:
        print(f'Hata: {e}')
        print(f'Genel mAP50: {metrics.seg.map50:.4f}')

In [ ]:
# ── ADIM 9: Palynofacies Analizi ─────────────────────────────
!pip install openpyxl -q

import glob

# best.pt'yi bul
best_models = glob.glob('/content/drive/MyDrive/palytoae/**/weights/best.pt', recursive=True)
best = sorted(best_models)[-1] if best_models else None
print(f'Model: {best}')

# Test görüntüleri
test_images = '/content/dataset_split/test/images'

# Çalıştır
!python /content/palynofacies.py \
    --weights {best} \
    --images  {test_images} \
    --out_dir /content/drive/MyDrive/palytoae/palynofacies_results